# Notebook 5: Sequential Testing and Group Sequential Methods

## Overview
This notebook addresses a critical practical problem: **the "peeking problem."** Many companies monitor A/B test results in real-time and stop early when they see promising results. This inflates Type I error unless we use proper sequential testing methods.

### Learning Objectives
- Understand why continuous monitoring inflates Type I error
- Learn how O'Brien-Fleming and Pocock boundaries work
- Implement sequential testing in practice
- Compare fixed-sample vs sequential designs

### The Core Problem
If you run an A/B test designed for 10,000 customers but check the result every 1,000 customers, your true Type I error is NOT 5%—it's much higher. Why? Because each peek is a hypothesis test, and if you do many tests, you'll eventually see p < 0.05 by chance alone.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try plotly
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb05', exist_ok=True)


In [2]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"Segments: {df['segment'].value_counts()}")

Data shape: (64000, 24)
Segments: segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


## Concept: The "Peeking Problem"

### The Core Issue
When you **continuously monitor** an A/B test and stop early when you see a favorable result, you're actually conducting multiple hypothesis tests. Each peek is an opportunity to make a Type I error.

### Mathematical Problem
- Single test at fixed sample size: Type I error = α = 0.05
- Two peeks at 50% and 100% of planned sample: Type I error ≈ 0.08 to 0.10 (depending on design)
- **Five peeks**: Type I error can exceed 0.15
- **Twenty peeks**: Type I error can exceed 0.20

This happens because the test statistics across peeks are **correlated but not perfectly correlated**, creating an opportunity to observe p < 0.05 by chance.

### Real-World Example
A company plans a 10,000-customer test with α = 0.05. Instead of waiting, they check:
- After 1,000 customers: p = 0.08 (not significant, but improving)
- After 2,000 customers: p = 0.06 (trending)
- After 3,000 customers: p = 0.04 (stop! declare victory!)

But they never would have seen p < 0.05 if they'd run all 10,000. They lucked out in the multiple comparisons.

### Solution: Sequential Testing
Use **pre-defined boundaries** that adjust the significance level at each peek to maintain overall Type I error control. The key methods are:

1. **O'Brien-Fleming**: Conservative early, liberal late (requires strong evidence early)
2. **Pocock**: Constant boundary (same threshold at each look)
3. **Spending functions**: Flexible allocation of alpha across looks (Lan-DeMets)

In [3]:
def simulate_fixed_sample_test(null_true=True, n_final=5000, conversion_rate_1=0.05, conversion_rate_2=0.05):
    """
    Simulate a fixed-sample A/B test, computing test statistics at multiple points.
    
    Parameters:
    -----------
    null_true : bool
        Whether the null hypothesis is truly true
    n_final : int
        Final sample size per group
    conversion_rate_1 : float
        True conversion rate for group 1
    conversion_rate_2 : float
        True conversion rate for group 2
    
    Returns:
    --------
    array : p-values at each peek (at 25%, 50%, 75%, 100% of target sample size)
    """
    peeks = np.array([0.25, 0.50, 0.75, 1.0])
    peek_sizes = (peeks * n_final).astype(int)
    
    p_values = []
    
    for peek_n in peek_sizes:
        # Generate data
        if null_true:
            conv1 = np.random.binomial(1, 0.05, peek_n)
            conv2 = np.random.binomial(1, 0.05, peek_n)
        else:
            conv1 = np.random.binomial(1, conversion_rate_1, peek_n)
            conv2 = np.random.binomial(1, conversion_rate_2, peek_n)
        
        # Two-proportion z-test
        p1, p2 = conv1.mean(), conv2.mean()
        se = np.sqrt(p1*(1-p1)/peek_n + p2*(1-p2)/peek_n)
        
        if se == 0:
            z_stat = 0
        else:
            z_stat = (p1 - p2) / se
        
        p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        p_values.append(p_val)
    
    return np.array(p_values)

# Simulate 10,000 null hypothesis tests under continuous monitoring (4 peeks)
print("Simulating 10,000 A/B tests with continuous monitoring...\n")

num_sims = 10000
peeks_matrix = np.zeros((num_sims, 4))

for i in range(num_sims):
    peeks_matrix[i, :] = simulate_fixed_sample_test(null_true=True, n_final=5000)

# Count how many tests would reject at each peek (p < 0.05)
rejections_per_peek = np.sum(peeks_matrix < 0.05, axis=0) / num_sims
ever_rejected = np.sum(np.any(peeks_matrix < 0.05, axis=1)) / num_sims

print("Type I Error by Peek Point:")
print("=" * 50)
for i, pct in enumerate([25, 50, 75, 100]):
    print(f"  After {pct}% of planned sample: {rejections_per_peek[i]:.3f}")

print(f"\nEver rejected (stopped early): {ever_rejected:.3f}")
print(f"\nTarget Type I error: 0.050")
print(f"Actual Type I error (any peek): {ever_rejected:.3f}")
print(f"\n*** Type I error inflation: {ever_rejected / 0.05:.1f}x ***")

Simulating 10,000 A/B tests with continuous monitoring...

Type I Error by Peek Point:
  After 25% of planned sample: 0.050
  After 50% of planned sample: 0.053
  After 75% of planned sample: 0.048
  After 100% of planned sample: 0.050

Ever rejected (stopped early): 0.186

Target Type I error: 0.050
Actual Type I error (any peek): 0.186

*** Type I error inflation: 3.7x ***


In [4]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## O'Brien-Fleming Spending Function

### The Idea
The O'Brien-Fleming method uses an **alpha spending function** that is:
- **Conservative early** (high threshold, hard to declare victory)
- **Liberal late** (low threshold, easier to declare victory)

This makes sense: early data has more uncertainty, so we require stronger evidence. Later data has accumulated and is more certain, so we can relax the threshold.

### The Formula
For K looks, the cumulative alpha spent at look k is:

**α_spend(t) = 2 × (1 - Φ(z_α/2 / √t))**

Where t = k/K (fraction of total looks completed).

### Example
For 4 looks with α = 0.05:
- Look 1 (25%): Boundary z = 2.413 (p ≈ 0.0158) — strong evidence needed
- Look 2 (50%): Boundary z = 2.050 (p ≈ 0.0404) — still stringent
- Look 3 (75%): Boundary z = 1.846 (p ≈ 0.0648) — relaxing
- Look 4 (100%): Boundary z = 1.960 (p ≈ 0.0500) — standard 0.05

This maintains overall Type I error at 5% while allowing early stopping.

In [5]:
def obrien_fleming_boundary(k, K, alpha=0.05):
    """
    Calculate O'Brien-Fleming boundary for look k out of K.
    
    Parameters:
    -----------
    k : int
        Current look (1, 2, ..., K)
    K : int
        Total number of looks
    alpha : float
        Overall Type I error rate
    
    Returns:
    --------
    float : Critical z-value for this look
    """
    t = k / K  # Information time
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_critical = z_alpha / np.sqrt(t)
    return z_critical

def pocock_boundary(k, K, alpha=0.05):
    """
    Calculate Pocock boundary (constant threshold adjusted for repeated testing).
    
    For Pocock, all looks have the same p-value threshold, but it's more stringent than 0.05.
    """
    # Pocock uses a fixed p-value threshold across all looks
    # For K looks, the boundary is approximately: p_threshold = alpha / (0.5 + 0.25*ln(K))
    # Here we use a standard approximation
    
    if K == 1:
        return stats.norm.ppf(1 - alpha / 2)
    elif K == 2:
        p_threshold = alpha / 2.045
    elif K == 3:
        p_threshold = alpha / 2.289
    elif K == 4:
        p_threshold = alpha / 2.447
    elif K == 5:
        p_threshold = alpha / 2.562
    else:
        # General approximation
        p_threshold = alpha / (0.5 + 0.25 * np.log(K))
    
    z_critical = stats.norm.ppf(1 - p_threshold / 2)
    return z_critical

# Calculate boundaries for 4 looks
K = 4
looks = np.arange(1, K + 1)

print("O'Brien-Fleming Boundaries (K=4 looks, α=0.05):")
print("=" * 60)
print(f"{'Look':<6} {'Information %':<15} {'Z Critical':<12} {'p-value':<12}")
print("-" * 60)

of_boundaries = []
pocock_boundaries = []

for k in looks:
    z_of = obrien_fleming_boundary(k, K, alpha=0.05)
    p_of = 2 * (1 - stats.norm.cdf(z_of))
    z_poc = pocock_boundary(k, K, alpha=0.05)
    p_poc = 2 * (1 - stats.norm.cdf(z_poc))
    
    of_boundaries.append(z_of)
    pocock_boundaries.append(z_poc)
    
    info_pct = (k / K) * 100
    print(f"{k:<6} {info_pct:<15.0f} {z_of:<12.3f} {p_of:<12.4f}")

print(f"\nPocock Boundaries (constant, K=4 looks, α=0.05):")
print("=" * 60)
print(f"{'Look':<6} {'Information %':<15} {'Z Critical':<12} {'p-value':<12}")
print("-" * 60)

for k, z_poc in zip(looks, pocock_boundaries):
    p_poc = 2 * (1 - stats.norm.cdf(z_poc))
    info_pct = (k / K) * 100
    print(f"{k:<6} {info_pct:<15.0f} {z_poc:<12.3f} {p_poc:<12.4f}")

O'Brien-Fleming Boundaries (K=4 looks, α=0.05):
Look   Information %   Z Critical   p-value     
------------------------------------------------------------
1      25              3.920        0.0001      
2      50              2.772        0.0056      
3      75              2.263        0.0236      
4      100             1.960        0.0500      

Pocock Boundaries (constant, K=4 looks, α=0.05):
Look   Information %   Z Critical   p-value     
------------------------------------------------------------
1      25              2.318        0.0204      
2      50              2.318        0.0204      
3      75              2.318        0.0204      
4      100             2.318        0.0204      


In [6]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Apply Sequential Testing to Hillstrom Data

We'll simulate as if the Hillstrom data arrived in 5 sequential batches. At each batch, we calculate the z-statistic for the conversion rate difference (Mens Email vs No Email) and compare to the boundaries.

In [7]:
# Prepare data for sequential analysis
# Split into 5 sequential looks
K_seq = 5
df_sorted = df.sort_values('recency').reset_index(drop=True)
n_total = len(df_sorted)
n_per_look = n_total // K_seq

print(f"Total observations: {n_total}")
print(f"Approximate per look: {n_per_look}\n")

# Track metrics at each look
looks_seq = np.arange(1, K_seq + 1)
z_statistics = []
p_values = []
info_times = []
n_cumulative = []

for k in looks_seq:
    # Get cumulative data up to this look
    n_cum = min(k * n_per_look, n_total)
    df_cum = df_sorted.iloc[:n_cum]
    
    # Filter to Mens Email and No Email
    mens_data = df_cum[df_cum['segment'] == "Mens E-Mail"]
    control_data = df_cum[df_cum['segment'] == "No E-Mail"]
    
    # Calculate conversion rates
    p1 = mens_data['conversion'].mean()
    p2 = control_data['conversion'].mean()
    n1, n2 = len(mens_data), len(control_data)
    
    # Z-test
    p_pooled = (mens_data['conversion'].sum() + control_data['conversion'].sum()) / n_cum
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    
    if se > 0:
        z_stat = (p1 - p2) / se
    else:
        z_stat = 0
    
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    info_time = k / K_seq
    
    z_statistics.append(z_stat)
    p_values.append(p_val)
    info_times.append(info_time)
    n_cumulative.append(n_cum)
    
    print(f"Look {k}: N={n_cum}, Men's Conv Rate={p1:.4f}, Control Conv Rate={p2:.4f}")
    print(f"        Z-statistic={z_stat:.3f}, p-value={p_val:.4f}\n")

# Calculate boundaries for K_seq looks
of_boundaries_seq = [obrien_fleming_boundary(k, K_seq) for k in looks_seq]
pocock_boundaries_seq = [pocock_boundary(k, K_seq) for k in looks_seq]

Total observations: 64000
Approximate per look: 12800

Look 1: N=12800, Men's Conv Rate=0.0180, Control Conv Rate=0.0088
        Z-statistic=4.496, p-value=0.0000

Look 2: N=25600, Men's Conv Rate=0.0159, Control Conv Rate=0.0083
        Z-statistic=5.542, p-value=0.0000

Look 3: N=38400, Men's Conv Rate=0.0137, Control Conv Rate=0.0074
        Z-statistic=6.033, p-value=0.0000

Look 4: N=51200, Men's Conv Rate=0.0133, Control Conv Rate=0.0065
        Z-statistic=7.739, p-value=0.0000

Look 5: N=64000, Men's Conv Rate=0.0125, Control Conv Rate=0.0057
        Z-statistic=9.037, p-value=0.0000



In [8]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

In [9]:
# Create summary comparison table
summary_data = []

for i, k in enumerate(looks_seq):
    summary_data.append({
        'Look': k,
        'Cumulative N': n_cumulative[i],
        'Z-Statistic': f"{z_statistics[i]:.3f}",
        'p-value': f"{p_values[i]:.4f}",
        'OF Boundary': f"{of_boundaries_seq[i]:.3f}",
        'OF Decision': 'STOP' if abs(z_statistics[i]) > of_boundaries_seq[i] else 'Continue',
        'Pocock Decision': 'STOP' if abs(z_statistics[i]) > pocock_boundaries_seq[i] else 'Continue'
    })

summary_table = pd.DataFrame(summary_data)

print("\n=== SEQUENTIAL TESTING SUMMARY ===\n")
print(summary_table.to_string(index=False))

# Save summary
summary_table.to_csv('../data/outputs/nb05/nb05_sequential_results.csv', index=False)

print("\n\nInterpretation:")
print("-" * 70)
print("OF = O'Brien-Fleming (conservative early, liberal late)")
print("Pocock = Constant boundary (same p-value at all looks)")
print("\nWith O'Brien-Fleming:")
print("  - Early looks require strong evidence (higher Z threshold)")
print("  - Later looks have lower threshold, allowing earlier stopping")
print("  - Overall Type I error still controlled at 5%")


=== SEQUENTIAL TESTING SUMMARY ===

 Look  Cumulative N Z-Statistic p-value OF Boundary OF Decision Pocock Decision
    1         12800       4.496  0.0000       4.383        STOP            STOP
    2         25600       5.542  0.0000       3.099        STOP            STOP
    3         38400       6.033  0.0000       2.530        STOP            STOP
    4         51200       7.739  0.0000       2.191        STOP            STOP
    5         64000       9.037  0.0000       1.960        STOP            STOP


Interpretation:
----------------------------------------------------------------------
OF = O'Brien-Fleming (conservative early, liberal late)
Pocock = Constant boundary (same p-value at all looks)

With O'Brien-Fleming:
  - Early looks require strong evidence (higher Z threshold)
  - Later looks have lower threshold, allowing earlier stopping
  - Overall Type I error still controlled at 5%


## Key Takeaways on Sequential Testing

### When to Use Sequential Testing
1. **Online experiments**: Results stream in gradually, desire to stop early if results are clear
2. **High cost of uncertainty**: When running the test is expensive (customer acquisition, computing)
3. **Ethical studies**: Medical trials where stopping early saves patients from ineffective treatments
4. **Business deadlines**: Need results faster than fixed-sample design allows

### When Fixed-Sample is Better
1. **Small data volume**: Few observations, limited opportunity to look
2. **Off-line analysis**: Data arrives in batch, can wait for pre-planned analysis
3. **Regulatory requirements**: Some contexts require pre-registered, fixed-sample designs

### Best Practices
1. **Pre-register the design**: Decide number of looks and boundaries BEFORE collecting data
2. **Use O'Brien-Fleming**: More powerful than Pocock for fixed sample size
3. **Monitor power**: Ensure you still have ~80% power to detect meaningful effect
4. **Consider adaptive designs**: Lan-DeMets spending functions allow flexible look times
5. **Be transparent**: Report that sequential testing was used and what boundaries were applied

### Common Mistakes
1. **Ignoring correlation**: Believing each peek is independent (it's not)
2. **Changing the boundary**: "Just this once" checking at unplanned times
3. **Multiple testing without adjustment**: Testing multiple metrics without correction
4. **Not adjusting for early stopping**: Reporting CI/estimates as if fixed sample

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [10]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb05")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb05


In [13]:
from scipy import stats as spstats
# Chart 1: Sequential z-stat trajectory with O'Brien-Fleming + Pocock boundaries
np.random.seed(42)
# Simulate the running z-statistic for treatment vs control conversion
treated = df_blog[df_blog["segment"] != "No E-Mail"].sample(frac=1, random_state=42).reset_index(drop=True)
ctrl = df_blog[df_blog["segment"] == "No E-Mail"].sample(frac=1, random_state=42).reset_index(drop=True)
N = min(len(treated), len(ctrl))
steps = np.linspace(500, N, 40).astype(int)
z_stats=[]
for n in steps:
    t = treated["conversion"].iloc[:n].values
    c = ctrl["conversion"].iloc[:n].values
    p1,p2 = t.mean(), c.mean()
    pool = (t.sum()+c.sum())/(2*n)
    se = np.sqrt(pool*(1-pool)*(2/n)) if pool*(1-pool)>0 else 1e-9
    z_stats.append((p1-p2)/se)
info_frac = steps / N
# O'Brien-Fleming (approx): z_k = z_alpha / sqrt(info_frac)
z_a = spstats.norm.ppf(1 - 0.025)
of_bound = z_a / np.sqrt(info_frac)
# Pocock (approx constant ~ 2.413 for alpha=.05 with many looks)
pocock_bound = np.full_like(info_frac, 2.413)
fixed_bound = np.full_like(info_frac, 1.96)

fig = go.Figure()
fig.add_trace(go.Scatter(x=info_frac, y=z_stats, mode="lines+markers",
                         line=dict(color="#4C8BB8", width=3),
                         marker=dict(size=7), name="Observed Z-statistic",
                         hovertemplate="Info frac: %{x:.2f}<br>Z: %{y:.2f}<extra></extra>"))
fig.add_trace(go.Scatter(x=info_frac, y=of_bound, mode="lines",
                         line=dict(color="#E74C3C", dash="dash"),
                         name="O'Brien-Fleming upper"))
fig.add_trace(go.Scatter(x=info_frac, y=-of_bound, mode="lines",
                         line=dict(color="#E74C3C", dash="dash"),
                         name="O'Brien-Fleming lower", showlegend=False))
fig.add_trace(go.Scatter(x=info_frac, y=pocock_bound, mode="lines",
                         line=dict(color="#F39C12", dash="dot"), name="Pocock upper"))
fig.add_trace(go.Scatter(x=info_frac, y=-pocock_bound, mode="lines",
                         line=dict(color="#F39C12", dash="dot"), name="Pocock lower", showlegend=False))
fig.add_hline(y=1.96, line_dash="longdash", line_color="#888",
              annotation_text="Fixed α=0.05 (±1.96)", annotation_position="top right")
fig.add_hline(y=-1.96, line_dash="longdash", line_color="#888")
fig.update_layout(**BASE_LAYOUT,
                  title="Sequential Z-Statistic with Stopping Boundaries",
                  xaxis=dict(title="Information Fraction (proportion of final sample)", automargin=True),
                  yaxis=dict(title="Z-Statistic", automargin=True), height=560,
                  legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
fig.write_html(os.path.join(OUT_DIR, "nb05_sequential_trajectory_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb05_sequential_trajectory_interactive.html")

# Chart 2: Spending function comparison
K = 5; ks = np.arange(1, K+1); info = ks/K
# Alpha spent under each function
def of_alpha_spent(t, alpha=0.05):
    return 2*(1 - spstats.norm.cdf(spstats.norm.ppf(1-alpha/2)/np.sqrt(t)))
def pocock_alpha_spent(t, alpha=0.05):
    return alpha * np.log(1 + (np.e - 1) * t)
def linear_alpha_spent(t, alpha=0.05):
    return alpha * t
fig = go.Figure()
tt = np.linspace(0.01, 1, 50)
fig.add_trace(go.Scatter(x=tt, y=[of_alpha_spent(t) for t in tt], mode="lines",
                         line=dict(color="#E74C3C", width=3), name="O'Brien-Fleming"))
fig.add_trace(go.Scatter(x=tt, y=[pocock_alpha_spent(t) for t in tt], mode="lines",
                         line=dict(color="#F39C12", width=3), name="Pocock"))
fig.add_trace(go.Scatter(x=tt, y=[linear_alpha_spent(t) for t in tt], mode="lines",
                         line=dict(color="#888", width=3, dash="dash"), name="Linear"))
fig.update_layout(**BASE_LAYOUT, title="Alpha Spending Functions",
                  xaxis=dict(title="Information Fraction", automargin=True),
                  yaxis=dict(title="Cumulative α Spent", automargin=True), height=480)
fig.write_html(os.path.join(OUT_DIR, "nb05_spending_functions_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb05_spending_functions_interactive.html")


  ✓ nb05_sequential_trajectory_interactive.html


  ✓ nb05_spending_functions_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb05/` so it can be dropped straight into the
blog post.


In [12]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# Sequential Testing — Results-Display Tables
from scipy import stats as _sp

K = 4
alpha_overall = 0.05
looks = np.arange(1, K+1)

def obf_z(k, K, alpha=0.05):
    z_K = _sp.norm.ppf(1 - alpha/2)
    return z_K * np.sqrt(K/k)

def pocock_z(k, K, alpha=0.05):
    # standard Pocock boundary approximation
    from math import log
    p_thr = alpha / (0.5 + 0.25 * np.log(K))
    return _sp.norm.ppf(1 - p_thr/2)

of_rows, poc_rows = [], []
for k in looks:
    z_of  = obf_z(k, K, alpha_overall)
    p_of  = 2 * (1 - _sp.norm.cdf(z_of))
    z_poc = pocock_z(k, K, alpha_overall)
    p_poc = 2 * (1 - _sp.norm.cdf(z_poc))
    info = f"{(k/K)*100:.0f}%"
    of_rows.append((k, info, f"{z_of:.3f}", f"{p_of:.4f}"))
    poc_rows.append((k, info, f"{z_poc:.3f}", f"{p_poc:.4f}"))

fig = table_card(
    "O'Brien–Fleming Boundaries (K = 4 looks, α = 0.05)",
    ["Look", "Information %", "Z critical", "Nominal p"],
    [[r[0] for r in of_rows], [r[1] for r in of_rows],
     [r[2] for r in of_rows], [r[3] for r in of_rows]],
    [100, 160, 160, 160])
fig.write_html(os.path.join(OUT_DIR, "nb05_obf_boundaries_interactive.html"), **PLOTLY_KW)
fig.show()

fig = table_card(
    "Pocock Boundaries (constant, K = 4 looks, α = 0.05)",
    ["Look", "Information %", "Z critical", "Nominal p"],
    [[r[0] for r in poc_rows], [r[1] for r in poc_rows],
     [r[2] for r in poc_rows], [r[3] for r in poc_rows]],
    [100, 160, 160, 160])
fig.write_html(os.path.join(OUT_DIR, "nb05_pocock_boundaries_interactive.html"), **PLOTLY_KW)
fig.show()

# Sequential testing summary card — when would we stop under each rule?
# Simulated: cumulative z at each look for Any Email vs Control on visit
any_email = df_blog[df_blog["segment"] != "No E-Mail"].reset_index(drop=True)
control   = df_blog[df_blog["segment"] == "No E-Mail"].reset_index(drop=True)
min_n = min(len(any_email), len(control))
chunk = min_n // K

summary = []
for k in looks:
    n = k * chunk
    a = any_email["visit"].iloc[:n]
    c = control["visit"].iloc[:n]
    p1, p2 = a.mean(), c.mean()
    pool = (a.sum()+c.sum())/(len(a)+len(c))
    se = np.sqrt(pool*(1-pool)*(1/len(a)+1/len(c)))
    z = (p1-p2)/se if se>0 else 0
    stop_of  = abs(z) > obf_z(k, K, alpha_overall)
    stop_poc = abs(z) > pocock_z(k, K, alpha_overall)
    summary.append((k, f"{n:,}", f"{z:.3f}",
                    "✓ stop" if stop_of else "continue",
                    "✓ stop" if stop_poc else "continue"))

fig = go.Figure(data=[go.Table(
    columnwidth=[80, 160, 160, 200, 200],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Look","Cumulative n (per arm)","Observed Z","O'Brien–Fleming decision","Pocock decision"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*summary)),
               fill_color=[stripe_col(len(summary))]*5,
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Sequential Testing Summary — Any Email vs Control (visit)",
                  height=36 + 30*len(summary) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb05_sequential_summary_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb05 sequential-testing cards saved")


  ✓ nb05 sequential-testing cards saved
